# Notebook 08 — Xây dựng và Đánh giá Mô hình

**Mục tiêu**: Train, evaluate và chọn mô hình dự báo doanh thu tốt nhất
- Baseline model
- XGBoost / LightGBM/ Catboost
- So sánh models theo MAE, RMSE, R², Adjusted R-Square

**Tương ứng báo cáo**: Phần 2 — Chương 6 (Xây dựng và đánh giá mô hình dự báo)

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_processed
from src.preprocessing import prepare_stl_dataframe
from src.evaluation import evaluate
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

FIG_DIR = Path('../outputs/figures/model')
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [2]:
sales      = load_processed('sales_clean.csv')
orders     = load_processed('orders_clean.csv')
order_items = load_processed('order_items_clean.csv')
products   = load_processed('products_clean.csv')
returns    = load_processed('returns_clean.csv')
web_traffic = load_processed('web_traffic_clean.csv')
payments   = load_processed('payments_clean.csv')
promotions = load_processed('promotions_clean.csv')

d:\KienTap_RevenueEcommerce\revenue_forecast_ecommerce\notebooks\..\src\data_loader.py:40: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(PROCESSED / subfolder / filename)


### 1. Hàm chia tách dữ liệu train test theo expanding window

In [ ]:
def create_expanding_window_splits(data):

    # =====================================================
    # PREP DATA
    # =====================================================

    dataset = (
        data[['LogRevenue']]
        .reset_index()
        .rename(columns={
            'date': 'ds',
            'LogRevenue': 'y'
        })
    )

    dataset['ds'] = pd.to_datetime(dataset['ds'])

    # =====================================================
    # SPLITS (EXPANDING WINDOW - 4 FOLDS)
    # =====================================================

    splits = [
    {'fold': 1, 'test_start': '2021-01-01', 'test_end': '2021-06-30'},
    {'fold': 2, 'test_start': '2021-07-01', 'test_end': '2021-12-31'},
    {'fold': 3, 'test_start': '2022-01-01', 'test_end': '2022-06-30'},  # FIX HERE
    {'fold': 4, 'test_start': '2022-07-01', 'test_end': '2022-12-31'}
]

    fold_data = []

    # =====================================================
    # BUILD FOLDS
    # =====================================================

    for split in splits:

        test_start = pd.to_datetime(split['test_start'])
        test_end = pd.to_datetime(split['test_end'])

        # train_end = day before test_start
        train_end = test_start - pd.Timedelta(days=1)

        # expanding train
        train = dataset[
            dataset['ds'] <= train_end
        ].reset_index(drop=True)

        test = dataset[
            (dataset['ds'] >= test_start) &
            (dataset['ds'] <= test_end)
        ].reset_index(drop=True)

        fold_data.append({
            'fold': split['fold'],
            'train_end': train_end,
            'train': train,
            'test': test
        })

        # =================================================
        # PRINT INFO
        # =================================================

        print(f"\n{'='*60}")
        print(f"FOLD {split['fold']}")
        print(f"{'='*60}")

        print(
            f"Train : {train['ds'].min().date()} -> {train['ds'].max().date()} "
            f"| Size = {len(train)}"
        )

        print(
            f"Test  : {test['ds'].min().date()} -> {test['ds'].max().date()} "
            f"| Size = {len(test)}"
        )

    return fold_data

### 2. Xây dựng mô hình Baseline

In [4]:
stl_df = prepare_stl_dataframe(sales)
stl_df.head()

,revenue,LogRevenue
date,,
2012-07-04,5123547.94,15.449358
2012-07-05,2751773.45,14.827757
2012-07-06,3054029.42,14.931973
2012-07-07,2667930.94,14.796814
2012-07-08,2360851.90,14.674534


In [6]:
stl_df = prepare_stl_dataframe(sales)

fold_splits = create_expanding_window_splits(stl_df)

fold_rows = []

for split in fold_splits:

    fold  = split['fold']
    train = split['train'].copy()
    test  = split['test'].copy()

    print(f"\n{'='*70}")
    print(f'FOLD {fold}')
    print(f"{'='*70}")

    train['ds'] = pd.to_datetime(train['ds'])
    test['ds']  = pd.to_datetime(test['ds'])

    train = train[['ds', 'y']]
    test  = test[['ds', 'y']]

    # y ở đây là LogRevenue → back-transform để evaluate trên revenue thật
    y_true_log = test['y'].values
    y_true_rev = np.expm1(y_true_log)

    # =====================================================
    # SEASONAL NAIVE BASELINE
    # =====================================================
    naive_preds_rev = []

    for ds in test['ds']:
        target_date = ds - pd.Timedelta(days=365)

        match = train[train['ds'] == target_date]['y']

        if len(match) > 0:
            naive_log = match.values[0]
        else:
            diff = (train['ds'] - target_date).abs()
            naive_log = train.loc[diff.idxmin(), 'y']

        naive_preds_rev.append(np.expm1(naive_log))

    y_pred_naive_rev = np.array(naive_preds_rev)

    metrics_naive = evaluate(
        y_true=y_true_rev,
        y_pred=y_pred_naive_rev,
        n_features=0,
        label=f'Fold {fold} - SEASONAL NAIVE'
    )

    # =====================================================
    # PROPHET
    # =====================================================
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.1,
        seasonality_mode='multiplicative'
    )

    model.fit(train)

    future = model.make_future_dataframe(
        periods=len(test),
        freq='D'
    )

    forecast = model.predict(future)

    forecast_test = (
        forecast[['ds', 'yhat']]
        .merge(test, on='ds', how='inner')
    )

    forecast_train = (
        forecast[['ds', 'yhat']]
        .merge(train, on='ds', how='inner')
    )

    y_pred_log = forecast_test['yhat'].values

    y_pred_raw_rev = np.expm1(y_pred_log)

    train_residuals = (
        forecast_train['y'].values
        - forecast_train['yhat'].values
    )

    smearing_factor = np.mean(
        np.exp(train_residuals)
    )

    y_pred_duan_rev = (
        np.expm1(y_pred_log)
        * smearing_factor
    )

    metrics_raw = evaluate(
        y_true=y_true_rev,
        y_pred=y_pred_raw_rev,
        n_features=1,
        label=f'Fold {fold} - PROPHET RAW'
    )

    metrics_duan = evaluate(
        y_true=y_true_rev,
        y_pred=y_pred_duan_rev,
        n_features=1,
        label=f'Fold {fold} - PROPHET DUAN'
    )

    for model_name, metrics in [
        ('Seasonal Naive', metrics_naive),
        ('Prophet RAW', metrics_raw),
        ('Prophet DUAN', metrics_duan)
    ]:

        fold_rows.append({
            'Fold': fold,
            'Model': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'R2': metrics['R2'],
            'Adj_R2': metrics['Adjusted_R2']
        })


    # =====================================================
    # PLOT
    # =====================================================
    plt.figure(figsize=(14, 5))
    plt.plot(test['ds'], y_true_rev,       label='Actual',          color='black')
    plt.plot(test['ds'], y_pred_naive_rev, label='Seasonal Naive',  color='gray',      linestyle='--')
    plt.plot(test['ds'], y_pred_raw_rev,   label='Prophet RAW',     color='steelblue', linestyle='--')
    plt.plot(test['ds'], y_pred_duan_rev,  label='Prophet DUAN',    color='darkorange',linestyle='--')
    plt.title(f'Baseline Comparison - Fold {fold} (Revenue Scale)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# =========================================================
# RESULT TABLE
# =========================================================
results = pd.DataFrame(fold_rows)

print("\n" + "="*70)
print("FINAL RESULTS - REVENUE SCALE")
print("="*70)
display(results)

print("\n" + "="*70)
print("AVERAGE PERFORMANCE COMPARISON")
print("="*70)
avg = (
    results
    .groupby('Model')[['MAE', 'RMSE', 'R2', 'Adj_R2']]
    .mean()
    .sort_values('RMSE')
)

# format cho dễ đọc (không scientific notation)
avg_display = avg.copy()
avg_display['MAE'] = avg_display['MAE'].apply(lambda x: f"{x:,.2f}")
avg_display['RMSE'] = avg_display['RMSE'].apply(lambda x: f"{x:,.2f}")
avg_display['R2'] = avg_display['R2'].apply(lambda x: f"{x:.4f}")
avg_display['Adj_R2'] = avg_display['Adj_R2'].apply(lambda x: f"{x:.4f}")

display(avg_display)

KeyError: 'ds'